<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase46.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 46 — frozen generic TFL v15 Arm C acquisition and portability preflight

Phase 45 correctly left Arm A blocked because the private Athena and independent-review evidence was not present. Phase 46 therefore advances a different required comparison branch: **Arm C, the existing frozen generic Transformation Fidelity Layer (TFL) v15**.

This notebook:

- verifies the exact Phase 45 public chain from Google Drive;
- resolves the public release tag `tfl-submission-freeze-v15` to an immutable Git commit;
- downloads and hashes the frozen release archive;
- inventories the release without modifying it;
- verifies the generic TFL audit schema and information-fate vocabulary from the frozen source;
- runs repository-native Python tests if they are present and runnable in the base environment;
- checks whether the frozen release exposes an unmodified execution path applicable to the neuroimaging evidence objects;
- reports a strict Arm C readiness result.

### Non-negotiable comparator rule

Arm C is the **frozen generic TFL v15**. This notebook does not add MRI-specific laterality, longitudinal-pairing, provenance/model, or TRACE rules to v15. If the release cannot consume the neuroimaging evidence without rule modification, Arm C remains not executed and the portability limitation is recorded.

That distinction preserves the paper's comparison: generic TFL v15 versus the later neuroimaging-specific method.


In [1]:
# CODE CELL 1/8 — Drive, constants, helpers
import os, re, csv, json, hashlib, shutil, tempfile, zipfile, time, subprocess, sys
import datetime as dt
from pathlib import Path
from collections import Counter, defaultdict

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='46_frozen_generic_tfl_v15_armC_acquisition_and_portability_preflight'

EXPECTED_P45_SHAREABLE='92221e7587ffd02dbd182b712c9b635f26e4b41b52c7776c957ba6bb756b6726'
EXPECTED_P45_CONTRACT='d80e7ede309fedca3e95341b299ae4756c9a6796ea2148c4abc0479f8ee235da'

REPO='SANGHATI23/ohdsi-fhir-omop-showcase-demo'
TAG='tfl-submission-freeze-v15'

TFL_FIELDS=[
 'transformation_id','source_resource_type','source_resource_id',
 'source_reference_or_path','source_value_or_code','target_omop_table',
 'target_omop_record_id','target_omop_field','mapping_rule_id',
 'fidelity_status','warning_code','transformation_version'
]
FATE_STATES=['PRESERVED','NORMALIZED','COLLAPSED','AMBIGUOUS','UNMAPPED','DROPPED','TRACEABILITY_LOSS']

MRI_DIMENSIONS={
 'anatomy_laterality':['laterality','left','right','anatomy'],
 'value_unit':['ucum','unit','mm3','cm3','value'],
 'longitudinal_pairing':['longitudinal','timepoint','baseline','12-month','pairing'],
 'algorithm_identity':['algorithm','device','software version','model version'],
 'provenance':['provenance','lineage','source_reference_or_path'],
 'reliability_trace':['trace state','review_required','caution','reliability'],
 'cardinality_idempotence':['duplicate','dropped','unexpected','idempot'],
}

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

if os.environ.get('PHASE46_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE46_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
    print('Offline Drive-mirror mode:',MYDRIVE)
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE46_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase46_'))
STAGE=WORK/'outputs'; STAGE.mkdir()
SRC=WORK/'tfl_v15'; SRC.mkdir()
OUT_BASE=Path(os.environ.get('PHASE46_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase46')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
LOCK_PATH=OUT_BASE/'ICHI2027_Phase46_TFL_V15_RELEASE_LOCK.json'
print('Phase 46 output:',OUT)


Mounted at /content/drive
Phase 46 output: /content/drive/MyDrive/ICHI2027/Phase46/run_20261001T022909_973218Z


In [2]:
# CODE CELL 2/8 — Verify Phase 45 input chain

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

def find_by_sha(stem,ext,digest):
    good=[]
    for root in ROOTS:
        for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
            dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
            for fn in files:
                if name_matches(fn,stem,ext):
                    p=Path(here)/fn
                    try:
                        if sha256(p)==digest: good.append(p)
                    except OSError: pass
    if not good:
        raise FileNotFoundError(f'No SHA-approved Drive copy found for {stem}{ext}')
    return sorted(good,key=str)[0]

P45=find_by_sha('ICHI2027_Phase45_SHAREABLE','.json',EXPECTED_P45_SHAREABLE)
C45=find_by_sha('ICHI2027_Phase45_LOGICAL_CONTRACT_CARRIED_FORWARD','.csv',EXPECTED_P45_CONTRACT)

p45=json.loads(P45.read_text('utf-8'))
with C45.open(newline='',encoding='utf-8-sig') as f:
    contract45=list(csv.DictReader(f))

assert p45['phase']=='45_private_athena_binding_gap_and_demographic_policy_revalidation'
assert p45['armA']['evidence_supported_gates']==0
assert p45['armA']['remaining_gates']==10
assert p45['armA']['official_micdm_load_executed'] is False
assert p45['armA']['official_omop_etl_executed'] is False
assert p45['frozen_original_generic_tfl_armC_executed'] is False
assert len(contract45)==8

print('PASS: exact Phase 45 chain verified.')
print('Arm A evidence-supported gates:',p45['armA']['evidence_supported_gates'],'/10')
print('Frozen generic TFL Arm C previously executed:',p45['frozen_original_generic_tfl_armC_executed'])


PASS: exact Phase 45 chain verified.
Arm A evidence-supported gates: 0 /10
Frozen generic TFL Arm C previously executed: False


In [3]:
# CODE CELL 3/8 — Resolve tag to immutable commit and download the frozen archive
import requests

session=requests.Session()
session.headers.update({'User-Agent':'ICHI2027-neuroimaging-fhir-omop-fidelity-phase46'})

def api_json(url):
    r=session.get(url,timeout=60)
    r.raise_for_status()
    return r.json()

lock_origin=''

if LOCK_PATH.exists():
    release_lock=json.loads(LOCK_PATH.read_text('utf-8'))
    assert release_lock['repository']==REPO
    assert release_lock['tag']==TAG
    commit_sha=release_lock['commit_sha']
    lock_origin='REUSED_EXISTING_DRIVE_LOCK'
else:
    ref=api_json(f'https://api.github.com/repos/{REPO}/git/ref/tags/{TAG}')
    obj=ref['object']
    if obj['type']=='commit':
        commit_sha=obj['sha']
    elif obj['type']=='tag':
        tag_obj=api_json(obj['url'])
        if tag_obj['object']['type']!='commit':
            raise RuntimeError('Frozen tag does not resolve directly to a commit.')
        commit_sha=tag_obj['object']['sha']
    else:
        raise RuntimeError('Unexpected Git tag object type: '+str(obj['type']))
    assert re.fullmatch(r'[0-9a-f]{40}',commit_sha)
    release_lock={
      'project':PROJECT,
      'phase_created':PHASE,
      'repository':REPO,
      'tag':TAG,
      'commit_sha':commit_sha,
      'created_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
      'comparator_rule':'Use frozen generic v15 without neuroimaging-specific rule modifications'
    }
    lock_origin='NEW_LOCK_PENDING_ARCHIVE_VALIDATION'

archive=WORK/'tfl_v15.zip'
url=f'https://api.github.com/repos/{REPO}/zipball/{commit_sha}'
r=session.get(url,timeout=120,headers={'Accept':'application/vnd.github+json'})
r.raise_for_status()
archive.write_bytes(r.content)
assert archive.stat().st_size>1000

archive_sha=sha256(archive)
with zipfile.ZipFile(archive) as z:
    bad=z.testzip()
    if bad: raise RuntimeError('Corrupt release archive member: '+bad)
    z.extractall(SRC)

tops=[p for p in SRC.iterdir() if p.is_dir()]
if len(tops)!=1:
    raise RuntimeError('Expected one top-level directory in frozen release archive.')
RELEASE_ROOT=tops[0]

release_lock['archive_sha256']=archive_sha
release_lock['archive_bytes']=archive.stat().st_size

print('Frozen tag:',TAG)
print('Immutable commit:',commit_sha)
print('Archive SHA-256:',archive_sha)
print('Extracted root:',RELEASE_ROOT.name)


Frozen tag: tfl-submission-freeze-v15
Immutable commit: a4b7265adabf096e84b957de068e24a5ab853425
Archive SHA-256: e729610f050217a2c62c0536309c7e3ac3508f39992d7ed46b1b70c9bc9478dc
Extracted root: SANGHATI23-ohdsi-fhir-omop-showcase-demo-a4b7265


In [4]:
# CODE CELL 4/8 — Inventory frozen release and locate TFL method evidence

TEXT_EXT={'.py','.json','.jsonl','.csv','.tsv','.md','.txt','.rst','.yaml','.yml','.toml','.ini','.ipynb'}
inventory=[]
text_cache={}

for p in sorted(RELEASE_ROOT.rglob('*')):
    if not p.is_file(): continue
    rel=str(p.relative_to(RELEASE_ROOT))
    size=p.stat().st_size
    inventory.append({
      'path':rel,
      'bytes':size,
      'sha256':sha256(p),
      'suffix':p.suffix.lower()
    })
    if p.suffix.lower() in TEXT_EXT and size<=5_000_000:
        try:
            text_cache[rel]=p.read_text('utf-8',errors='replace')
        except Exception:
            pass

all_text='\n'.join(text_cache.values())
all_text_lower=all_text.lower()

schema_rows=[]
for field in TFL_FIELDS:
    paths=[rel for rel,txt in text_cache.items() if field.lower() in txt.lower()]
    schema_rows.append({
      'audit_field':field,
      'present_in_frozen_release':bool(paths),
      'evidence_file_count':len(paths),
      'example_evidence_path':paths[0] if paths else ''
    })

fate_rows=[]
for state in FATE_STATES:
    paths=[rel for rel,txt in text_cache.items() if state in txt]
    fate_rows.append({
      'fidelity_state':state,
      'present_in_frozen_release':bool(paths),
      'evidence_file_count':len(paths),
      'example_evidence_path':paths[0] if paths else ''
    })

schema_complete=all(r['present_in_frozen_release'] for r in schema_rows)
fate_complete=all(r['present_in_frozen_release'] for r in fate_rows)

version_evidence=[]
for rel,txt in text_cache.items():
    if '1.0.0' in txt and ('transformation' in txt.lower() or 'tfl' in txt.lower()):
        version_evidence.append(rel)

print('Release files:',len(inventory))
print('Canonical TFL fields found:',sum(r['present_in_frozen_release'] for r in schema_rows),'/',len(schema_rows))
print('Fidelity states found:',sum(r['present_in_frozen_release'] for r in fate_rows),'/',len(fate_rows))
print('Schema version 1.0.0 evidence files:',len(version_evidence))


Release files: 78
Canonical TFL fields found: 12 / 12
Fidelity states found: 7 / 7
Schema version 1.0.0 evidence files: 4


In [5]:
# CODE CELL 5/8 — Static neuroimaging portability audit without adding rules

portability=[]
for dimension,terms in MRI_DIMENSIONS.items():
    hits={}
    for term in terms:
        paths=[rel for rel,txt in text_cache.items() if term.lower() in txt.lower()]
        hits[term]=paths
    present_terms=[t for t,p in hits.items() if p]
    evidence_paths=sorted({p for ps in hits.values() for p in ps})
    portability.append({
      'neuroimaging_dimension':dimension,
      'search_terms':' | '.join(terms),
      'generic_v15_text_evidence_terms_found':' | '.join(present_terms),
      'evidence_file_count':len(evidence_paths),
      'example_evidence_path':evidence_paths[0] if evidence_paths else '',
      'interpretation':'STATIC_TEXT_EVIDENCE_ONLY_NOT_RULE_EXECUTION'
    })

# Stronger explicit neuroimaging terms.
explicit_neuro_terms=['ImagingStudy','IMAGE_OCCURRENCE','IMAGE_FEATURE','hippocamp','entorhin','laterality','TRACE_STATE','REVIEW_REQUIRED']
explicit_hits={}
for term in explicit_neuro_terms:
    explicit_hits[term]=[rel for rel,txt in text_cache.items() if term.lower() in txt.lower()]

explicit_neuro_rule_evidence=sum(bool(v) for v in explicit_hits.values())

print('Explicit neuroimaging term families found in frozen v15:',explicit_neuro_rule_evidence,'/',len(explicit_neuro_terms))
for term,paths in explicit_hits.items():
    print(term,':',len(paths))
print('No result here is treated as an executed comparator.')


Explicit neuroimaging term families found in frozen v15: 1 / 8
ImagingStudy : 1
IMAGE_OCCURRENCE : 0
IMAGE_FEATURE : 0
hippocamp : 0
entorhin : 0
laterality : 0
TRACE_STATE : 0
REVIEW_REQUIRED : 0
No result here is treated as an executed comparator.


In [6]:
# CODE CELL 6/8 — Execute repository-native Python tests when present; inspect runnable interfaces

test_files=[p for p in RELEASE_ROOT.rglob('*.py') if p.name.startswith('test_') or p.name.endswith('_test.py')]
pytest_status='NO_PYTHON_TESTS_FOUND'
pytest_returncode=None
pytest_tail=''

if test_files:
    try:
        proc=subprocess.run(
          [sys.executable,'-m','pytest','-q'],
          cwd=str(RELEASE_ROOT),
          capture_output=True,text=True,timeout=300
        )
        pytest_returncode=proc.returncode
        pytest_status='PASS' if proc.returncode==0 else 'FAILED_OR_DEPENDENCY_BLOCKED'
        pytest_tail=(proc.stdout+'\n'+proc.stderr)[-4000:]
    except subprocess.TimeoutExpired:
        pytest_status='TIMEOUT'
        pytest_tail='pytest exceeded 300 seconds'
    except Exception as e:
        pytest_status='ERROR'
        pytest_tail=repr(e)

# Evidence of callable/runnable interfaces, without guessing how to feed MRI data.
entrypoint_evidence=[]
for rel,txt in text_cache.items():
    low=txt.lower()
    flags=[]
    if 'if __name__' in low and '__main__' in low: flags.append('python_main')
    if 'argparse' in low: flags.append('argparse')
    if 'console_scripts' in low or '[project.scripts]' in low: flags.append('package_script')
    if 'def main(' in low: flags.append('main_function')
    if flags:
        entrypoint_evidence.append({'path':rel,'interface_evidence':'|'.join(flags)})

# Search for explicit instructions that connect TFL to arbitrary/new resource types.
generic_extension_evidence=[]
for rel,txt in text_cache.items():
    low=txt.lower()
    if ('resource type' in low or 'source_resource_type' in low) and ('rule' in low or 'mapping' in low):
        generic_extension_evidence.append(rel)

print('Repository Python test files:',len(test_files))
print('Native pytest status:',pytest_status)
print('Runnable-interface evidence files:',len(entrypoint_evidence))
print('Generic resource/rule documentation files:',len(generic_extension_evidence))
if pytest_tail:
    print('--- pytest tail ---')
    print(pytest_tail)


Repository Python test files: 0
Native pytest status: NO_PYTHON_TESTS_FOUND
Runnable-interface evidence files: 1
Generic resource/rule documentation files: 11


In [7]:
# CODE CELL 7/8 — Strict Arm C readiness decision

# Arm C is NOT considered executed merely because the release exists, its tests pass,
# or its schema is generic. It must actually expose an unmodified way to process the
# neuroimaging evidence through the frozen generic rules.

explicit_imaging_support = any(explicit_hits[t] for t in [
    'ImagingStudy','IMAGE_OCCURRENCE','IMAGE_FEATURE','hippocamp','entorhin','laterality'
])

if not schema_complete or not fate_complete:
    armc_status='BLOCKED_FROZEN_RELEASE_METHOD_ARTIFACTS_INCOMPLETE'
    armc_reason='Canonical generic TFL schema/fate vocabulary could not be fully evidenced from the frozen release.'
elif explicit_imaging_support:
    armc_status='PREFLIGHT_REQUIRES_DOCUMENTED_UNMODIFIED_EXECUTION_PATH'
    armc_reason='Frozen release contains some imaging-related text evidence, but static evidence does not establish that the generic v15 rules executed on this neuroimaging study.'
else:
    armc_status='PORTABILITY_BLOCKED_NO_EVIDENCED_UNMODIFIED_NEUROIMAGING_RULE_PATH'
    armc_reason='Frozen generic v15 was acquired intact, but the release does not evidence an unmodified neuroimaging-specific execution path. Adding MRI rules would create the proposed method, not Arm C.'

armc_executed=False

readiness=[
 {'criterion':'frozen_release_tag_resolved_to_commit','passed':bool(commit_sha),'detail':commit_sha},
 {'criterion':'release_archive_hash_frozen','passed':bool(archive_sha),'detail':archive_sha},
 {'criterion':'canonical_tfl_sidecar_schema_evidenced','passed':schema_complete,'detail':f"{sum(r['present_in_frozen_release'] for r in schema_rows)}/{len(schema_rows)} fields"},
 {'criterion':'canonical_information_fate_vocabulary_evidenced','passed':fate_complete,'detail':f"{sum(r['present_in_frozen_release'] for r in fate_rows)}/{len(fate_rows)} states"},
 {'criterion':'repository_native_tests_passed_if_present','passed':pytest_status in {'PASS','NO_PYTHON_TESTS_FOUND'},'detail':pytest_status},
 {'criterion':'unmodified_neuroimaging_execution_path_evidenced','passed':False,'detail':armc_status},
 {'criterion':'frozen_generic_tfl_armC_executed','passed':False,'detail':'Static acquisition/preflight is not execution'}
]

print('Arm C execution status:',armc_status)
print('Arm C executed:',armc_executed)
print(armc_reason)


Arm C execution status: PREFLIGHT_REQUIRES_DOCUMENTED_UNMODIFIED_EXECUTION_PATH
Arm C executed: False
Frozen release contains some imaging-related text evidence, but static evidence does not establish that the generic v15 rules executed on this neuroimaging study.


In [8]:
# CODE CELL 8/8 — Public-safe Phase 46 artifacts and Drive save

# Persist the release lock only after archive extraction and inventory succeeded.
OUT_BASE.mkdir(parents=True,exist_ok=True)
if not LOCK_PATH.exists():
    temp_lock=WORK/'release_lock.json'
    temp_lock.write_text(json.dumps(release_lock,sort_keys=True,indent=2)+'\n',encoding='utf-8')
    shutil.copyfile(temp_lock,LOCK_PATH)
    assert sha256(temp_lock)==sha256(LOCK_PATH)
    lock_origin='NEW_VALIDATED_LOCK_CREATED'
else:
    persisted=json.loads(LOCK_PATH.read_text('utf-8'))
    assert persisted['commit_sha']==commit_sha
    if 'archive_sha256' in persisted:
        assert persisted['archive_sha256']==archive_sha
    lock_origin='REUSED_EXISTING_DRIVE_LOCK'

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

artifacts=[]
artifacts.append(save('ICHI2027_Phase46_TFL_V15_FILE_INVENTORY.csv',inventory))
artifacts.append(save('ICHI2027_Phase46_TFL_SCHEMA_EVIDENCE.csv',schema_rows))
artifacts.append(save('ICHI2027_Phase46_TFL_FATE_VOCABULARY_EVIDENCE.csv',fate_rows))
artifacts.append(save('ICHI2027_Phase46_NEUROIMAGING_PORTABILITY_PREFLIGHT.csv',portability))
artifacts.append(save('ICHI2027_Phase46_ARM_C_READINESS.csv',readiness))

entry_path=STAGE/'ICHI2027_Phase46_RUNNABLE_INTERFACE_EVIDENCE.csv'
write_csv(entry_path,entrypoint_evidence or [{'path':'','interface_evidence':'NONE_FOUND'}])
artifacts.append(entry_path)

lock_copy=STAGE/'ICHI2027_Phase46_TFL_V15_RELEASE_LOCK.json'
lock_copy.write_text(json.dumps(release_lock,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(lock_copy)

readme=STAGE/'ICHI2027_Phase46_README.txt'
readme.write_text(
  'Phase 46 acquires and cryptographically freezes the generic TFL v15 comparator release.\n'
  'It inventories the unmodified frozen release, checks the canonical TFL sidecar fields and information-fate vocabulary, and runs native Python tests when available.\n'
  'Static text/schema evidence is not counted as execution of Arm C.\n'
  'MRI-specific rules are not added to the frozen release. Doing so would turn the comparator into the proposed neuroimaging-specific method.\n'
  'Arm A remains blocked by the Phase 45 evidence state and is not executed here.\n',
  encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase45_shareable_sha256':sha256(P45),
 'repository':REPO,
 'frozen_tag':TAG,
 'frozen_commit_sha':commit_sha,
 'frozen_archive_sha256':archive_sha,
 'release_lock_origin':lock_origin,
 'release_file_count':len(inventory),
 'canonical_tfl_fields_present':sum(r['present_in_frozen_release'] for r in schema_rows),
 'canonical_tfl_fields_total':len(schema_rows),
 'fidelity_states_present':sum(r['present_in_frozen_release'] for r in fate_rows),
 'fidelity_states_total':len(fate_rows),
 'schema_version_1_0_0_evidence_files':len(version_evidence),
 'repository_python_test_files':len(test_files),
 'repository_native_pytest_status':pytest_status,
 'explicit_neuroimaging_term_families_found':explicit_neuro_rule_evidence,
 'explicit_neuroimaging_term_families_total':len(explicit_neuro_terms),
 'armC_status':armc_status,
 'armC_reason':armc_reason,
 'frozen_original_generic_tfl_armC_executed':False,
 'armA_evidence_supported_gates_carried_forward':p45['armA']['evidence_supported_gates'],
 'armA_remaining_gates_carried_forward':p45['armA']['remaining_gates'],
 'official_micdm_load_executed':False,
 'official_omop_etl_executed':False,
 'real_authorized_mri_processed':False,
 'externally_independent_challenge_authored':False,
 'full_A_B_C_D_evaluation':False,
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Frozen-release acquisition, static compatibility inspection, and native release tests do not by themselves constitute execution of the generic TFL comparator on the neuroimaging evidence.'
}
summary_path=STAGE/'ICHI2027_Phase46_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase46_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2: raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase 46 frozen TFL v15 acquisition/preflight completed.')
print('Arm C:',armc_status)
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase 46 frozen TFL v15 acquisition/preflight completed.
Arm C: PREFLIGHT_REQUIRES_DOCUMENTED_UNMODIFIED_EXECUTION_PATH
Shareable: /content/drive/MyDrive/ICHI2027/Phase46/run_20261001T022909_973218Z/ICHI2027_Phase46_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase46/run_20261001T022909_973218Z/ICHI2027_Phase46_PUBLIC_OUTPUTS.zip


## Interpretation

A successful Phase 46 run can freeze and verify the **actual generic TFL v15 comparator**, but it must not automatically convert that into a claim that Arm C ran on the neuroimaging study.

If the frozen release has no documented unmodified path for `ImagingStudy` / MI-CDM neuroimaging evidence, that is itself an important comparator result: generic transformation-fidelity auditing exists, but the neuroimaging study requires new domain-specific rules. The subsequent implementation must keep the frozen generic comparator separate from the proposed neuroimaging-specific extension.
